# **Libraries**

In [ ]:
!pip install -U google-cloud-aiplatform lightgbm joblib pandas scikit-learn
!pip install --upgrade --force-reinstall google-auth==2.35.0

  Using cached google_auth-2.63.1-py3-none-any.whl.metadata (6.0 kB)
Using cached google_auth-2.63.1-py3-none-any.whl (271 kB)
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.35.0
    Uninstalling google-auth-2.35.0:
      Successfully uninstalled google-auth-2.35.0
  Using cached google_auth-2.35.0-py2.py3-none-any.whl.metadata (4.7 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pyasn1-0.6.5-py3-none-any.whl.metadata (8.4 kB)
Using cached google_auth-2.35.0-py2.py3-none-any.whl (208 kB)
Using cached cachetools-5.5.2-py3-none-any.whl (10 kB)
Using cached rsa-4.9.1-py3-none-any.whl (34 kB)
Using cached pyasn1-0.6.5-py3-none-any.whl (84 kB)
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
  Attempting uninstall: pyasn1
    Found existing installation: pyasn1 0.6.5
    Uninsta

In [ ]:
import os
import json
import joblib
import lightgbm as lgb
import pandas as pd
from google.cloud import bigquery
import subprocess
from google.cloud import aiplatform
from google.cloud import aiplatform
from google.cloud.aiplatform_v1.services.endpoint_service import EndpointServiceClient
from google.cloud.aiplatform_v1.types import Endpoint as EndpointType, PredictRequestResponseLoggingConfig
from google.cloud.aiplatform_v1.types.io import BigQueryDestination
from google.protobuf import field_mask_pb2

# **Upload Model to GCS**

In [ ]:
model = lgb.Booster(model_file = 'model_artifacts/loss_cost_lgbm.txt')
model

In [ ]:
project_id =  'gen-lang-client-0275567139'
region = "northamerica-northeast2"

bucket = f"{project_id}-loss-cost-models"
model_dir = "lgbm-model"

aiplatform.init(
    project=project_id,
    location=region
)
!gcloud storage cp --recursive model_artifacts gs://{bucket}/{model_dir}/

Copying file://model_artifacts/loss_cost_lgbm.txt to gs://gen-lang-client-0275567139-loss-cost-models/lgbm-model/model_artifacts/loss_cost_lgbm.txt
Copying file://model_artifacts/.ipynb_checkpoints/loss_cost_lgbm-checkpoint.txt to gs://gen-lang-client-0275567139-loss-cost-models/lgbm-model/model_artifacts/.ipynb_checkpoints/loss_cost_lgbm-checkpoint.txt
  Completed files 2/2 | 143.9kiB/143.9kiB                                      

Average throughput: 197.6MiB/s


# Grant the Workbench, Vertex AI and Cloud Build Service Accounts the appropriate  permissions.

In [ ]:
ls

Dockerfile                                         features.py
End_to_End_Loss_Cost_Model_Vertex_AI_Deploy.ipynb  model_artifacts/
api.py                                             requirements.txt
config.py


In [ ]:
%cat Dockerfile

# Dockerfile

FROM python:3.11-slim

# LightGBM's native core needs the OpenMP runtime at load time.
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 \
    && rm -rf /var/lib/apt/lists/*
    
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api:app", "--host=0.0.0.0", "--port=8000"]

In [ ]:
%cat api.py

import os
import json
import pandas as pd
import lightgbm as lgb
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from features import build_features

app = FastAPI(title = 'End_to_End_Loss_Cost_Model')

model_path = "/app/model_artifacts/loss_cost_lgbm.txt"

model = lgb.Booster(model_file = model_path)

class PredictionRequest(BaseModel):
    instances: list

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(request: PredictionRequest):

    try:
        df = pd.DataFrame(request.instances)
        df = build_features(df)
        
        preds = model.predict(df)
        engineered_features = df.to_dict(orient="records")

        result = {"predictions":  [
                {
                    "loss_cost": float(pred),
                    "engineered_features": feats,
                }
                for pred, feats in zip(preds, engineered_features)
            ]}
        return result

    except Exception as

# **Build and Push Docker image to Artifact Registry**

In [ ]:
repo = "loss-cost-repository"
image_name = "loss-cost-lgbm"
image_uri = f"{region}-docker.pkg.dev/{project_id}/{repo}/{image_name}:v3"

!gcloud builds submit --tag {image_uri} .

Creating temporary archive of 358 file(s) totalling 45.8 MiB before compression.
Uploading tarball of [.] to [gs://gen-lang-client-0275567139_cloudbuild/source/1795642617.559342-3a4dec3bb7c74b57bd9d97acabc8b765.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/gen-lang-client-0275567139/locations/global/builds/928fdfdd-c43f-4df4-b2d3-70c0c4c1fb9d].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/928fdfdd-c43f-4df4-b2d3-70c0c4c1fb9d?project=974642558409 ].
Waiting for build to complete. Polling interval: 1 second(s).
----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "928fdfdd-c43f-4df4-b2d3-70c0c4c1fb9d" using release legacy

FETCHSOURCE
Fetching storage object: gs://gen-lang-client-0275567139_cloudbuild/source/1795642617.559342-3a4dec3bb7c74b57bd9d97acabc8b765.tgz#1795642620083396
BUILD
Already have image (with digest): gcr.io/cloud-builders/gcb-internal
DEPRECATED: The legacy builder is deprecated and 

# **Register Model and Deploy to Vertex AI Endpoint**

In [ ]:
model = aiplatform.Model.upload(
    display_name="loss-cost-model-v3",
    artifact_uri=f"gs://{bucket}/{model_dir}/",
    serving_container_image_uri=image_uri,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8000]
)

Creating Model
Create Model backing LRO: projects/974642558409/locations/northamerica-northeast2/models/8431816023832788992/operations/8195643674965573632
Model created. Resource name: projects/974642558409/locations/northamerica-northeast2/models/8431816023832788992@1
To use this Model in another session:
model = aiplatform.Model('projects/974642558409/locations/northamerica-northeast2/models/8431816023832788992@1')


In [ ]:
endpoint = model.deploy(
    deployed_model_display_name="loss-cost-production-model",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=2,
    traffic_percentage=100,
    sync = False
)

Creating Endpoint
Create Endpoint backing LRO: projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272/operations/7829726205241720832
Endpoint created. Resource name: projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272')
Deploying model to Endpoint : projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272
Deploy Endpoint model backing LRO: projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272/operations/6203926739760971776
Endpoint model deployed. Resource name: projects/974642558409/locations/northamerica-northeast2/endpoints/5922763474596790272


# **BigQuery**

# We will persist prediction requests, loss cost model predictions and engineered features to BigQuery.

In [ ]:
bq_client = bigquery.Client(project=project_id)

dataset_id = "loss_cost_vertex_logs"
dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
dataset_ref.location = region

try:
    bq_client.create_dataset(dataset_ref, exists_ok=True)
    print(f"Dataset {dataset_id} ready")
except Exception as e:
    print(e)

Dataset loss_cost_vertex_logs ready


In [ ]:
client_options = {"api_endpoint": f"{region}-aiplatform.googleapis.com"}
endpoint_client = EndpointServiceClient(client_options=client_options)

table = f"bq://{project_id}.loss_cost_vertex_logs.lgbm_predictions"

logging_config = PredictRequestResponseLoggingConfig(
    enabled=True,
    sampling_rate=1.0,
    bigquery_destination=BigQueryDestination(output_uri=table),
)

endpoint_proto = EndpointType(
    name=endpoint.resource_name,
    predict_request_response_logging_config=logging_config,
)

update_mask = field_mask_pb2.FieldMask(
    paths=["predict_request_response_logging_config"]
)

endpoint_client.update_endpoint(
    endpoint=endpoint_proto,
    update_mask=update_mask,
)

ep = endpoint_client.get_endpoint(name=endpoint.resource_name)
cfg = ep.predict_request_response_logging_config
print("Enabled:", cfg.enabled)
print("Sampling rate:", cfg.sampling_rate)
print("Destination:", cfg.bigquery_destination.output_uri)

Enabled: True
Sampling rate: 1.0
Destination: bq://gen-lang-client-0275567139.loss_cost_vertex_logs.lgbm_predictions


# **Test Endpoint**

In [ ]:
test_instance = pd.DataFrame([{"RiskArea": 5.0, "BonusMalus": 100, "DrivAge": 35, "LicAge": 120, "VehUsage": "Private"}])
test_instance = test_instance.iloc[[0]].to_dict(orient="records")
print(test_instance)

[{'RiskArea': 5.0, 'BonusMalus': 100, 'DrivAge': 35, 'LicAge': 120, 'VehUsage': 'Private'}]


In [ ]:
response = endpoint.predict(
    instances=test_instance
)

print(response.predictions)

[{'engineered_features': {'bm_to_licage': 0.0, 'bm_to_age': 0.0, 'licage_log': 2.397895272798371, 'first_two_years': 0.0, 'bm_penality_x_risk_area': 0.0, 'VehUsage_lgb': 'Private', 'bm_bonus': 0.0, 'late_starter': 1.0, 'young': 0.0, 'first_year': 0.0, 'bm_x_age': 0.0, 'bm_penality': 0.0, 'senior': 0.0, 'bm_centered': 0.0, 'bm_is_bonus': 0.0, 'driv_age_sq': 1225.0, 'bm_is_penality': 0.0, 'young_x_licAge_fst_2': 0.0, 'inv_driv_age': 0.02857142857142857, 'young_x_risk_area': 0.0, 'experience_gap': 7.0, 'RiskArea': 5.0, 'licage_years': 10.0}, 'loss_cost': 570.4891651846364}]


# **Check BigQuery Log**

In [ ]:
query = f"""
SELECT
  logging_time,
  request_id,
  deployed_model_id,
  request_payload,
  response_payload
FROM `{project_id}.loss_cost_vertex_logs.lgbm_predictions`
ORDER BY logging_time DESC
LIMIT 10
"""

df = bq_client.query(query).to_dataframe()
print(df)

                      logging_time                     request_id  \
0 2026-11-25 21:52:49.119897+00:00  4226277673528513174.000000000   

     deployed_model_id                                    request_payload  \
0  3515114484744060928  [{"VehUsage":"Private","BonusMalus":100,"DrivA...   

                                    response_payload  
0  [{"engineered_features":{"RiskArea":5,"VehUsag...  


In [ ]:
df['request_payload'][0]

array(['{"VehUsage":"Private","BonusMalus":100,"DrivAge":35,"LicAge":120,"RiskArea":5}'],
      dtype=object)

In [ ]:
df['response_payload'][0]

array(['{"engineered_features":{"RiskArea":5,"VehUsage_lgb":"Private","bm_bonus":0,"bm_centered":0,"bm_is_bonus":0,"bm_is_penality":0,"bm_penality":0,"bm_penality_x_risk_area":0,"bm_to_age":0,"bm_to_licage":0,"bm_x_age":0,"driv_age_sq":1225,"experience_gap":7,"first_two_years":0,"first_year":0,"inv_driv_age":0.028571428571428571,"late_starter":1,"licage_log":2.3978952727983711,"licage_years":10,"senior":0,"young":0,"young_x_licAge_fst_2":0,"young_x_risk_area":0},"loss_cost":570.48916518463636}'],
      dtype=object)